In [2]:
import pandas as pd

# Load all your monthly sheets — each has a "Totals" sheet with same columns
# Just update the file names as per what you have
df_m1_1 = pd.read_excel("KE-26-0126-CGD-SUPE-Y26-M1-1.xlsx", sheet_name= "Totals")
df_m12_12 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M12-12.xlsx", sheet_name= "Totals")
df_m11_11 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M11-11.xlsx", sheet_name= "Totals")
df_m10_10 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M10-10.xlsx", sheet_name="Totals")
df_m9_9 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M9-9.xlsx", sheet_name="Totals")
df_m9_8 = pd.read_excel("KE-25-0122-CGD-SUPE-Y25-M8-8.xlsx", sheet_name="Totals")
df_m9_7 = pd.read_excel("KE-25-0122-CGD-SUPE-Y25-M7-7.xlsx", sheet_name="Totals")


# Optional: Add a Month column if not already there (just for identification)
df_m1_1["Month"] ="January"
df_m12_12["Month"] = "December"
df_m11_11["Month"] = "November"
df_m10_10["Month"] = "October"
df_m9_9["Month"] = "September"
df_m9_8["Month"] = "August"
df_m9_7["Month"] = "July" 


# Combine them all into one DataFrame
combined_df = pd.concat([df_m9_7, df_m9_8, df_m9_9,df_m10_10,df_m11_11,df_m12_12,df_m1_1], ignore_index=True)

# Save combined dataset
combined_df.to_excel("Combined_data.xlsx", index=False)

print("✅ Combined all months successfully!")
print(f"📊 Total rows combined: {len(combined_df)}")
print("🗂️ File saved as Combined_Totals.xlsx")


✅ Combined all months successfully!
📊 Total rows combined: 9273
🗂️ File saved as Combined_Totals.xlsx


In [3]:
import pandas as pd
import re

# --- Load dataset ---
df = pd.read_excel("Combined_data.xlsx")

# --- Check required columns ---
required_cols = {"Sub Family Name", "Item Name"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"❌ The file must have columns: {required_cols}")

# --- Define combined cleaning logic ---
def clean_subfamily(row):

    # Normalize input
    subfamily = str(row["Sub Family Name"]).strip().lower()
    item_name = str(row["Item Name"]).strip().lower()

    if "b/wash" in item_name or "s/gel" in item_name:
        return "body wash"
    if "soap bar" in item_name:
        return "bar soap"
    if "topex bleach twin" in item_name:
        return "regular bleach"

    # --- SAFISHA SPECIFIC FIXES (NO HANDWASH) ---
    if item_name.startswith("safisha"):

        # ALL SURFACE CLEANER
        if "all surf" in item_name or "all surface" in item_name:
            return "m/purpose"

        # MULTI FLOOR CLEANER
        if "mult flr" in item_name or "multi flr" in item_name or "mult floor" in item_name:
            return "m/purpose floor"

        # BATHROOM & SHOWER
        if "bathrm" in item_name or "bathroom" in item_name or "shower" in item_name:
            return "detergent bathroom"

        # REGULAR BLEACH
        if "bleach" in item_name:
            return "regular bleach" 

        # DISINFECTANT / ANTIBACTERIAL
        if "disinfectant" in item_name or "anti-bact" in item_name or "antibac" in item_name:
            return "antiseptic"

        # KITCHEN CLEANER
        if "kitchen" in item_name:
            return "detergent kitchen"

    # Default fallback to ensure cleaned is always defined
    cleaned = subfamily

    # --------------------------
    # 1️⃣ Apply detailed cleaning first
    # --------------------------

    # --- CLOTH BLEACH ---
    if re.search(r"cl[o]?a?th\s*bleach", subfamily):
        if re.search(r"colou?r", item_name):
            cleaned = "bleach colors"
        elif "home dry cleaner" in item_name:
            cleaned = "m/purpose floor"
        elif "softener" in item_name:
            cleaned = "concentrate"
        else:
            cleaned = "regular bleach"

    # --- DILUTE / CONCENTRATE ---
    elif "dilute" in subfamily:
        if "topex" in item_name:
            cleaned = "regular bleach"
        elif any(word in item_name for word in ["conc", "conce", "concentrate"]):
            cleaned = "concentrate"
        else:
            cleaned = "dilute"
    elif "concentrate" in subfamily:
        cleaned = "concentrate"

    # --- MULTIPURPOSE ---
    elif "m/purpose" in subfamily or "multipurpose" in subfamily:
        if "floor" in item_name:
            cleaned = "m/purpose floor"
        elif "kitchen" in item_name:
            cleaned = "detergent kitchen"
        else:
            cleaned = "m/purpose"

    # --- LIQUID SOAP BEAUTY ---
    elif "liquid soap beauty" in subfamily:
        if any(word in item_name for word in ["h/wash", "handwash"]):
            cleaned = "liquid soap beauty"
        elif "bathrm" in item_name or "shower" in item_name:
            cleaned = "detergent bathroom"
        elif re.search(r"a\.?bact|antibact|a/bacterial|a\.b|antibac", item_name):
            cleaned = "liquid soap antibacterial"
        elif "bleach" in item_name:
            cleaned = "regular bleach"
        elif "kitchen" in item_name:
            cleaned = "detergent kitchen"
        elif "disinfectant" in item_name:
            cleaned = "antiseptic"
        elif "bar" in item_name:
            cleaned = "bar soap"
        elif "b/wash" in item_name:
            cleaned = "body wash"
        else:
            cleaned = "liquid soap beauty"

    # --- LIQUID SOAP SPECIALITIES ---
    elif "liquid soap specialities" in subfamily:
        if "b/wash" in item_name:
            cleaned = "body wash"
        elif "bar" in item_name:
            cleaned = "bar soap"
        else:
            cleaned = "liquid soap specialities"

    # --- DISINFECTANTS & ANTISEPTIC ---
    elif "disinfectants" in subfamily or re.search(r"dis[iy]n?f[eai]ct[a-z]*", subfamily) or re.search(r"dis[iy]n?f[eai]ct[a-z]*", item_name):
        if "unigel" in item_name:
            cleaned = "m/purpose floor"
        else:
            cleaned = "disinfectants"
    elif "antiseptic" in subfamily:
        cleaned = "antiseptic"

    # --- SCOURING ---
    elif "scouring" in subfamily or "scour" in subfamily:
        if any(word in item_name for word in ["mult-liquid", "liquid", "liq", "multipurpose"]):
            cleaned = "multipurpose dishwashing"
        elif any(word in item_name for word in ["scourer", "scouring"]):
            cleaned = "scouring powders"
        else:
            cleaned = "dishwashing liq. & paste"

    # --- DISHWASHING LIQ & PASTE ---
    elif "dishwashing liq. & paste" in subfamily:
        if "multipurpose" in item_name:
            cleaned = "multipurpose dishwashing"
        elif any (word in item_name for word in ["paste","pasta"]):
            cleaned = "dishwashing Paste"
        elif any(word in item_name for word in ["dishwash", "dish wash", "d/wash", "d.wash", "washing", "d/liquid", "d/w", "mdw", "wash", "dwl"]):
            cleaned = "dishwashing"
        elif any(word in item_name for word in ["t/clnr", "t/c", "domestos"]):
            cleaned = "wc liquid"
        elif "h/wash" in item_name:
            cleaned = "handwash"
        else:
            cleaned = "dishwashing liq. & paste"

    # --- GLASS & WINDOW CLEANERS ---
    elif "glass" in subfamily or "window" in item_name:
        if "300ml" in item_name or "320ml" in item_name:
            cleaned = "window cleaner"
        elif "lens" in item_name or "wipes" in item_name:
            cleaned = "others"
        else:
            cleaned = "glass cleaner"

    # --- WC LIQUID / KITCHEN ---
    if any(word in item_name for word in ["t/clnr", "tc", "domestos", "t.gl", "toilet cleaner"]) or "wc liquid" in subfamily:
        if "kitchen" in item_name:
            cleaned = "detergent kitchen"
        else:
            cleaned = "wc liquid"

    # --- WATER GUARD ---
    elif "other" in subfamily and "water purifier" in item_name:
        cleaned = "water guard"

    # --------------------------
    # 2️⃣ Combine categories AFTER cleaning
    # --------------------------
    cleaned_lower = str(cleaned).strip().lower()
    original_lower = str(row["Sub Family Name"]).strip().lower()

    # Fabric Softeners
    if cleaned_lower in ["dilute", "concentrate"] or original_lower in ["dilute", "concentrate"]:
        return "fabric softeners"

    # Handwash
    if cleaned_lower in ["liquid soap beauty", "liquid soap antibacterial", "liquid soap specialities", "body wash"] \
       or original_lower in ["liquid soap beauty", "liquid soap antibacterial", "liquid soap specialities"]:
        return "handwash"

    return cleaned_lower

# --- Apply cleaning logic ---
df["Cleaned Sub Family Name"] = df.apply(clean_subfamily, axis=1)

# --- Summary ---
changed = (df["Cleaned Sub Family Name"] != df["Sub Family Name"].str.lower()).sum()
print(f"✅ Cleaned {changed} rows out of {len(df)} total.")

# --- Save output files ---
df.to_excel("final_data_totals.xlsx", index=False)
df[df["Cleaned Sub Family Name"] != df["Sub Family Name"].str.lower()].to_excel("changed_rows.xlsx", index=False)

print("✅ Cleaning complete — saved as 'final_data_totals.xlsx' and 'changed_rows.xlsx'")


✅ Cleaned 5945 rows out of 9273 total.
✅ Cleaning complete — saved as 'final_data_totals.xlsx' and 'changed_rows.xlsx'


In [4]:
import pandas as pd

# Load all your monthly sheets — each has a "Totals" sheet with same columns
# Just update the file names as per what you have
df_m1_1 = pd.read_excel("KE-26-0126-CGD-SUPE-Y26-M1-1.xlsx", sheet_name= "Branches")
df_m12_12 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M12-12.xlsx", sheet_name= "Branches")
df_m11_11 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M11-11.xlsx", sheet_name= "Branches")
df_m10_10 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M10-10.xlsx", sheet_name="Branches")
df_m9_9 = pd.read_excel("KE-25-0169-CGD-SUPE-Y25-M9-9.xlsx", sheet_name="Branches")
df_m9_8 = pd.read_excel("KE-25-0122-CGD-SUPE-Y25-M8-8.xlsx", sheet_name="Branches")
df_m9_7 = pd.read_excel("KE-25-0122-CGD-SUPE-Y25-M7-7.xlsx", sheet_name="Branches")


# Optional: Add a Month column if not already there (just for identification)
df_m1_1["Month"] ="January"
df_m12_12["Month"] = "December"
df_m11_11["Month"] = "November"
df_m10_10["Month"] = "October"
df_m9_9["Month"] = "September"
df_m9_8["Month"] = "August"
df_m9_7["Month"] = "July"


# Combine them all into one DataFrame
combined_df = pd.concat([df_m9_7, df_m9_8, df_m9_9,df_m10_10, df_m11_11, df_m12_12,df_m1_1 ], ignore_index=True)
# Save combined dataset
combined_df.to_excel("Combined_Branches.xlsx", index=False)

print("✅ Combined all months successfully!")
print(f"📊 Total rows combined: {len(combined_df)}")
print("🗂️ File saved as Combined_Totals.xlsx")


✅ Combined all months successfully!
📊 Total rows combined: 159826
🗂️ File saved as Combined_Totals.xlsx


In [5]:
import pandas as pd
import re

# --- Load dataset ---
df = pd.read_excel("Combined_Branches.xlsx")

# --- Check required columns ---
required_cols = {"Sub Family Name", "Item Name"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"❌ The file must have columns: {required_cols}")

# --- Define combined cleaning logic ---
def clean_subfamily(row):

    # Normalize input
    subfamily = str(row["Sub Family Name"]).strip().lower()
    item_name = str(row["Item Name"]).strip().lower()

    if "b/wash" in item_name or "s/gel" in item_name:
        return "body wash"
    if "soap bar" in item_name:
        return "bar soap"
    if "topex bleach twin" in item_name:
        return "regular bleach"

    # --- SAFISHA SPECIFIC FIXES (NO HANDWASH) ---
    if item_name.startswith("safisha"):

        # ALL SURFACE CLEANER
        if "all surf" in item_name or "all surface" in item_name:
            return "m/purpose"

        # MULTI FLOOR CLEANER
        if "mult flr" in item_name or "multi flr" in item_name or "mult floor" in item_name:
            return "m/purpose floor"

        # BATHROOM & SHOWER
        if "bathrm" in item_name or "bathroom" in item_name or "shower" in item_name:
            return "detergent bathroom"

        # REGULAR BLEACH
        if "bleach" in item_name:
            return "regular bleach"

        # DISINFECTANT / ANTIBACTERIAL
        if "disinfectant" in item_name or "anti-bact" in item_name or "antibac" in item_name:
            return "antiseptic"

        # KITCHEN CLEANER
        if "kitchen" in item_name:
            return "detergent kitchen"

    # Default fallback to ensure cleaned is always defined
    cleaned = subfamily

    # --------------------------
    # 1️⃣ Apply detailed cleaning first
    # --------------------------

    # --- CLOTH BLEACH ---
    if re.search(r"cl[o]?a?th\s*bleach", subfamily):
        if re.search(r"colou?r", item_name):
            cleaned = "bleach colors"
        elif "home dry cleaner" in item_name:
            cleaned = "m/purpose floor"
        elif "softener" in item_name:
            cleaned = "concentrate"
        else:
            cleaned = "regular bleach"

    # --- DILUTE / CONCENTRATE ---
    elif "dilute" in subfamily:
        if "topex" in item_name:
            cleaned = "regular bleach"
        elif any(word in item_name for word in ["conc", "conce", "concentrate"]):
            cleaned = "concentrate"
        else:
            cleaned = "dilute"
    elif "concentrate" in subfamily:
        cleaned = "concentrate"

    # --- MULTIPURPOSE ---
    elif "m/purpose" in subfamily or "multipurpose" in subfamily:
        if "floor" in item_name:
            cleaned = "m/purpose floor"
        elif "kitchen" in item_name:
            cleaned = "detergent kitchen"
        else:
            cleaned = "m/purpose"

    # --- LIQUID SOAP BEAUTY ---
    elif "liquid soap beauty" in subfamily:
        if any(word in item_name for word in ["h/wash", "handwash"]):
            cleaned = "liquid soap beauty"
        elif "bathrm" in item_name or "shower" in item_name:
            cleaned = "detergent bathroom"
        elif re.search(r"a\.?bact|antibact|a/bacterial|a\.b|antibac", item_name):
            cleaned = "liquid soap antibacterial"
        elif "bleach" in item_name:
            cleaned = "regular bleach"
        elif "kitchen" in item_name:
            cleaned = "detergent kitchen"
        elif "disinfectant" in item_name:
            cleaned = "antiseptic"
        elif "bar" in item_name:
            cleaned = "bar soap"
        elif "b/wash" in item_name:
            cleaned = "body wash"
        else:
            cleaned = "liquid soap beauty"

    # --- LIQUID SOAP SPECIALITIES ---
    elif "liquid soap specialities" in subfamily:
        if "b/wash" in item_name:
            cleaned = "body wash"
        else:
            cleaned = "liquid soap specialities"

    # --- DISINFECTANTS & ANTISEPTIC ---
    elif "disinfectants" in subfamily or re.search(r"dis[iy]n?f[eai]ct[a-z]*", subfamily) or re.search(r"dis[iy]n?f[eai]ct[a-z]*", item_name):
        if "unigel" in item_name:
            cleaned = "m/purpose floor"
        else:
            cleaned = "disinfectants"
    elif "antiseptic" in subfamily:
        cleaned = "antiseptic"

    # --- SCOURING ---
    elif "scouring" in subfamily or "scour" in subfamily:
        if any(word in item_name for word in ["mult-liquid", "liquid", "liq", "multipurpose"]):
            cleaned = "multipurpose dishwashing"
        elif any(word in item_name for word in ["scourer", "scouring"]):
            cleaned = "scouring powders"
        else:
            cleaned = "dishwashing liq. & paste"

    # --- DISHWASHING LIQ & PASTE ---
    elif "dishwashing liq. & paste" in subfamily:
        if "multipurpose" in item_name:
            cleaned = "multipurpose dishwashing"
        elif any (word in item_name for word in ["paste","pasta"]):
            cleaned = "dishwashing Paste"
        elif any(word in item_name for word in ["dishwash", "dish wash", "d/wash", "d.wash", "washing", "d/liquid", "d/w", "mdw", "wash", "dwl"]):
            cleaned = "dishwashing"
        elif any(word in item_name for word in ["t/clnr", "t/c", "domestos"]):
            cleaned = "wc liquid"
        elif "h/wash" in item_name:
            cleaned = "handwash"
        else:
            cleaned = "dishwashing liq. & paste"

    # --- GLASS & WINDOW CLEANERS ---
    elif "glass" in subfamily or "window" in item_name:
        if "300ml" in item_name or "320ml" in item_name:
            cleaned = "window cleaner"
        elif "lens" in item_name or "wipes" in item_name:
            cleaned = "others"
        else:
            cleaned = "glass cleaner"

    # --- WC LIQUID / KITCHEN ---
    if any(word in item_name for word in ["t/clnr", "tc", "domestos", "t.gl", "toilet cleaner"]) or "wc liquid" in subfamily:
        if "kitchen" in item_name:
            cleaned = "detergent kitchen"
        else:
            cleaned = "wc liquid"

    # --- WATER GUARD ---
    elif "other" in subfamily and "water purifier" in item_name:
        cleaned = "water guard"

    # --------------------------
    # 2️⃣ Combine categories AFTER cleaning
    # --------------------------
    cleaned_lower = str(cleaned).strip().lower()
    original_lower = str(row["Sub Family Name"]).strip().lower()

    # Fabric Softeners
    if cleaned_lower in ["dilute", "concentrate"] or original_lower in ["dilute", "concentrate"]:
        return "fabric softeners"

    # Handwash
    if cleaned_lower in ["liquid soap beauty", "liquid soap antibacterial", "liquid soap specialities", "body wash"] \
       or original_lower in ["liquid soap beauty", "liquid soap antibacterial", "liquid soap specialities"]:
        return "handwash"

    return cleaned_lower

# --- Apply cleaning logic ---
df["Cleaned Sub Family Name"] = df.apply(clean_subfamily, axis=1)

# --- Summary ---
changed = (df["Cleaned Sub Family Name"] != df["Sub Family Name"].str.lower()).sum()
print(f"✅ Cleaned {changed} rows out of {len(df)} total.")

# --- Save output files ---
df.to_excel("Combined_Branches.xlsx", index=False)
df[df["Cleaned Sub Family Name"] != df["Sub Family Name"].str.lower()].to_excel("changed_rows.xlsx", index=False)

print("✅ Cleaning complete — saved as 'final_data_totals.xlsx' and 'changed_rows.xlsx'")


✅ Cleaned 109122 rows out of 159826 total.
✅ Cleaning complete — saved as 'final_data_totals.xlsx' and 'changed_rows.xlsx'


In [6]:
import pandas as pd

# Normalization function
def normalize_branch_code(x):
    if pd.isna(x):
        return x
    x = str(x).upper().strip()
    x = x.replace(" ", "")        # remove spaces
    x = x.replace("--", "-")      # fix double dashes
    x = x.replace("_", "-")       # fix underscores
    parts = x.split("-")
    if len(parts) >= 2:
        return parts[0] + "-" + parts[1]
    return x

# Mapping dictionary (only normalized keys)
store_mapping = {
    "3110-KRIV": "TWO RIVERS",
    "3111-KHUB": "HUB",
    "3112-KBBY": "EASTLEIGH",
    "3118-KSOF": "SOUTHFIELD",
    "3119-KIRU": "RUIRU",
    "3120-KTRM": "TRM",
    "3123-KMGA": "MEGA",
    "3126-KGLR": "GALERIA",
    "3133-KKI1": "KISUMU UNITED",
    "8114-KXGCM": "GARDEN CITY",
    "8115-KXWGT": "WESTGATE",
    "8116-KXNGN": "NEXGEN",
    "8117-KXVYA": "VALLEY ARCADE",
    "8119-KXKLM": "KILIMANI",
    "8121-KXKEC": "ST ELLIES",
    "8122-KJCN": "JUNCTION",
    "8123-KXCTH": "COMET HOUSE",
    "8124-KSRT": "SARIT",
    "8128-KXVLM": "VILLAGE MARKET",
    "8130-XKED": "WARRIS RUAI",
    "8131-KXNCM": "NYALI",
    "8139-KXDIN": "DIANI",
    "8140-XKEF": "PROMENADE MOMBASA",
    "8142-XKEH": "RUNDA",
    "8144-XKEL": "RAPTHA PROMENADE",
    "8145-XKEK": "SEA ANGELS",
    "8148-XKEI": "RUBIS MAKUTANO",
    "8149-XKES": "RONGAI",
    "8151-XKEG": "GTC",
    "8156-XKEQ": "SPRING VALLEY"
}

# Load your file
df = pd.read_excel("Combined_Branches.xlsx")

# Normalize Branch column
df["Branch"] = df["Branch"].astype(str).apply(normalize_branch_code)

# Add store names based on mapping
df["Store Name"] = df["Branch"].map(store_mapping).fillna("No Match Found")

# Save back to same file (as you requested)
df.to_excel("Combined_Branches.xlsx", index=False)

print("✅ Store names successfully added to Combined_Branches.xlsx")


✅ Store names successfully added to Combined_Branches.xlsx


In [7]:
import pandas as pd

# --- Mapping: Branch → Handler(s) ---
branch_handler_map = {
    "RUIRU": "PAUL GUCHU",
    "PROMENADE MOMBASA": "ELIAS NTHIGA NDWIGA",
    "ST ELLIES": "ELIZABETH MUMBUA, VICTOR MWAKA",
    "RUNDA": "JULIET NDUTA MBURU",
    "TRM": "CLEFTON WAMBUA MUENDO",
    "VALLEY ARCADE": "SHAMITA MOHAMED",
    "PARKLANDS": "JAMES KARUGU WAITHERA, ZAKAYO VINCENT",
    "NYALI": "ELIAS NTHIGA NDWIGA",
    "DIANI": "VERONICA ANYONA WOCHETE",
    "EASTLEIGH": "PHYLLIS WANJIKU, VICTOR MWAKA",
    "GALERIA": "DENNIS KIPRUTO",
    "KILIMANI": "ANTONY MUTUNE NZAU",
    "KISUMU UNITED": "WALUBENGO VICTOR",
    "COMET HOUSE": "ELIZABETH MUMBUA, VICTOR MWAKA",
    "MEGA": "JAMES KARUGU WAITHERA",
    "NEXGEN": "JAMES KARUGU WAITHERA",
    "RAPTHA PROMENADE": "FRANCIS NDEGWA KIRING'U",
    "RONGAI": "IRENE NTHENYA MUTHOKA",
    "SARIT": "ELIZABETH MUMBUA, FRANCIS NDEGWA KIRING'U",
    "SOUTHFIELD": "CYNTHIA ANYANGO",
    "SPRING VALLEY": "ELIZABETH MUMBUA, VICTOR MWAKA, ZAKAYO VINCENT",
    "HUB": "ANTONY MUTUNE NZAU, DENNIS KIPRUTO, JOEL SIMIYU",
    "TWO RIVERS": "ZAKAYO VINCENT",
    "WARRIS RUAI": "EMILY MORAA",
    "WESTGATE": "ELIZABETH MUMBUA, VICTOR MWAKA, ZAKAYO VINCENT",
    "SEA ANGELS": "VERONICA ANYONA WOCHETE",
    "GARDEN CITY": "CLEFTON WAMBUA MUENDO",
    "JUNCTION": "ANTONY MUTUNE NZAU",
    "VILLAGE MARKET": "ZAKAYO VINCENT",
    "GTC": "ELIZABETH MUMBUA, VICTOR MWAKA, ZAKAYO VINCENT"
}

# --- Load the branches file ---
df = pd.read_excel("Combined_Branches.xlsx")

# Normalize branch names (optional but recommended)
df["Store Name"] = df["Store Name"].str.strip().str.upper()

# --- Map handlers ---
df["Handler"] = df["Store Name"].map(branch_handler_map).fillna("No Match Found")

# --- Save result ---
df.to_excel("Branches_With_Handlers.xlsx", index=False)

print("✅ Handlers successfully mapped and saved to 'Combined_Branches_With_Handlers.xlsx'")


✅ Handlers successfully mapped and saved to 'Combined_Branches_With_Handlers.xlsx'


In [8]:
import pandas as pd

# --- STEP 1: Define Branch → Region mapping ---
branch_region_map = {
    # Nairobi Region
    "RUIRU": "Nairobi Region",
    "ST ELLIES": "Nairobi Region",
    "RUNDA": "Nairobi Region",
    "TRM": "Nairobi Region",
    "VALLEY ARCADE": "Nairobi Region",
    "PARKLANDS": "Nairobi Region",
    "EASTLEIGH": "Nairobi Region",
    "GALERIA": "Nairobi Region",
    "KILIMANI": "Nairobi Region",
    "COMET HOUSE": "Nairobi Region",
    "MEGA": "Nairobi Region",
    "NEXGEN": "Nairobi Region",
    "RAPTHA PROMENADE": "Nairobi Region",
    "RONGAI": "Nairobi Region",
    "SARIT": "Nairobi Region",
    "SOUTHFIELD": "Nairobi Region",
    "SPRING VALLEY": "Nairobi Region",
    "HUB": "Nairobi Region",
    "TWO RIVERS": "Nairobi Region",
    "WARRIS RUAI": "Nairobi Region",
    "WESTGATE": "Nairobi Region",
    "SEA ANGELS": "Nairobi Region",
    "GARDEN CITY": "Nairobi Region",
    "JUNCTION": "Nairobi Region",
    "VILLAGE MARKET": "Nairobi Region",
    "GTC": "Nairobi Region",

    # Coast Region
    "PROMENADE MOMBASA": "Coast Region",
    "NYALI": "Coast Region",
    "DIANI": "Coast Region",

    # Western / Nyanza Region
    "KISUMU UNITED": "Western/Nyanza Region",

    #Central region.
    "RUBIS MAKUTANO": "Central Region"
}

# --- STEP 2: Load your data ---
df = pd.read_excel("Branches_With_Handlers.xlsx")

# --- STEP 3: Normalize Branch names ---
df["Store Name"] = df["Store Name"].str.strip().str.upper()

# --- STEP 4: Map Region ---
df["Region"] = df["Store Name"].map(branch_region_map).fillna("No Region Found")

# --- STEP 5: Save updated file ---
df.to_excel("Final_data_branches.xlsx", index=False)

print("✅ Regions successfully mapped and saved to 'Combined_Branches_With_Regions.xlsx'")


✅ Regions successfully mapped and saved to 'Combined_Branches_With_Regions.xlsx'
